Imports

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package punkt to C:\Users\YASHASVI
[nltk_data]     JADAV\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\YASHASVI
[nltk_data]     JADAV\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\YASHASVI
[nltk_data]     JADAV\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Load Dataset

In [2]:
DATA_PATH = "../data/raw/AMAZON_FASHION.json"

df = pd.read_json(DATA_PATH, lines=True)

print(f"Dataset loaded with {df.shape[0]} reviews")


Dataset loaded with 883636 reviews


Keep ONLY Required Columns

In [3]:
df = df[
    [
        "reviewerID",
        "asin",
        "overall",
        "reviewText",
        "summary",
        "verified",
        "unixReviewTime",
        "reviewTime",
        "vote"
    ]
]


Remove Empty Reviews

In [4]:
df = df[df["reviewText"].notnull()].reset_index(drop=True)

print(f"Remaining reviews after removing null text: {df.shape[0]}")


Remaining reviews after removing null text: 882403


Remove duplicates

In [5]:
before = len(df)
df = df.drop_duplicates(subset=["reviewerID", "reviewText"])
after = len(df)
print(f"Removed {before-after} duplicate reviews")


Removed 31040 duplicate reviews


Timestamp conversion

In [6]:
df["reviewTime"] = pd.to_datetime(df["reviewTime"], errors='coerce')


Helpful vote handling

In [7]:
df["vote"] = df["vote"].fillna(0)
df["vote"] = pd.to_numeric(df["vote"], errors="coerce").fillna(0)


Define Minimal Cleaning Function

In [8]:
def clean_text(text):
    text = str(text).lower()
    # remove html
    text = re.sub(r'<.*?>', '', text)
    # remove punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # tokenize
    tokens = word_tokenize(text)
    # remove stopwords
    tokens = [w for w in tokens if w not in stop_words]
    # lemmatization
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)


Apply Cleaning to Review Text

In [9]:
df["clean_review_text"] = df["reviewText"].apply(clean_text)


Verify Before vs After

In [10]:
df[["reviewText", "clean_review_text"]].sample(3)


,reviewText,clean_review_text
516885,Small,small
284793,Seriously so in love with this. It is so cute ...,seriously love cute one favorite outfit
819049,Goooood,goooood


Save Cleaned Dataset

In [11]:
OUTPUT_PATH = "../data/processed/reviews_clean.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("Cleaned dataset saved to data/processed/reviews_clean.csv")


Cleaned dataset saved to data/processed/reviews_clean.csv
